In [1]:
# Assignment 5 - RAG Agent for Remote Work

In [2]:
!pip install -q langchain-openai langchain-community chromadb tiktoken sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 

In [6]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()

print("OpenRouter key loaded")

OpenRouter key loaded


In [7]:
documents = [
    "Remote work gives employees more flexibility in managing their schedules and work-life balance.",
    "Working remotely can reduce commuting time and transportation costs for employees.",
    "Remote work can increase productivity for some employees by reducing office distractions.",
    "Companies can save money on office space, utilities, and other overhead costs when teams work remotely.",
    "Remote work can improve access to global talent because companies are not limited to hiring in one location.",
    "One challenge of remote work is communication difficulty, especially when teams work across time zones.",
    "Remote workers may feel isolated or disconnected from their colleagues and company culture.",
    "Managing performance remotely can be harder for team leaders without clear processes and expectations.",
    "Cybersecurity risks may increase in remote work environments if employees use insecure networks or devices.",
    "Successful remote work often depends on strong digital tools, clear communication, and regular check-ins."
]

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20
)

docs = text_splitter.create_documents(documents)

print("Number of chunks:", len(docs))
print(docs[0])

Number of chunks: 10
page_content='Remote work gives employees more flexibility in managing their schedules and work-life balance.'


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded")

/tmp/ipykernel_293/1455820253.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded


In [10]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

print("Vector store created")

Vector store created


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Retriever ready")

Retriever ready


In [12]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("Model ready")

Model ready


In [13]:
question = "What are the benefits and challenges of remote work?"

retrieved_docs = retriever.invoke(question)

print("Retrieved Documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)
    print("-" * 50)

Retrieved Documents:

Document 1:
Remote work gives employees more flexibility in managing their schedules and work-life balance.
--------------------------------------------------

Document 2:
Remote work can increase productivity for some employees by reducing office distractions.
--------------------------------------------------

Document 3:
Successful remote work often depends on strong digital tools, clear communication, and regular check-ins.
--------------------------------------------------


In [14]:
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""

response = model.invoke(prompt)

print("Final Answer:")
print(response.content)

Final Answer:
**Benefits (as stated in the context)**  
- Remote work gives employees more flexibility in managing their schedules and work‑life balance.  
- It can increase productivity for some employees by reducing office distractions.  
- Successful remote work depends on strong digital tools, clear communication, and regular check‑ins (these are presented as essential requirements for making remote work work well).

**Challenges (as stated in the context)**  
- The context does not explicitly mention any challenges associated with remote work.
